# Sovereign Driver QLoRA T4x2

QLoRA fine-tune of `Qwen/Qwen3-4B-Instruct-2507` on the agent-loop trace corpus using **both
T4 GPUs** via DistributedDataParallel, followed by a held-out benchmark against
the untouched base model.

Set the accelerator to **GPU T4 x2** and turn **Internet on** before running.

Three things this gets right that a naive port does not:

| | why it matters on a T4 |
|---|---|
| `fp16` + gradient scaler | Turing (sm_75) has no bf16 — asking for it fails at model load |
| `attn_implementation="sdpa"` | FlashAttention-2 needs Ampere (sm_80+) |
| `device_map={"": LOCAL_RANK}` | pins a whole model per rank; `"auto"` shards one model across both cards and deadlocks under DDP |

Quality settings target the failure the previous three adapters hit — a
training loss of 0.0016 on 226 traces, which is memorisation. `max_len=2048`
keeps the whole corpus instead of dropping 63% of it, one epoch replaces three,
and the best checkpoint is chosen by **eval** loss with early stopping.

## 1. Environment — check the accelerator BEFORE anything expensive

A previous run failed here in a way worth preventing: Kaggle assigned a single
**Tesla P100 (sm_60)**, the notebook launched `torchrun --nproc_per_node=2`
anyway, and rank 1 indexed a GPU that did not exist. It died with
`invalid device ordinal` — *after* downloading 8 GB of weights.

The accelerator cannot be requested through the API. `machine_shape` silently
normalises any unknown value back to a generic `Gpu`, so **GPU T4 x2 must be
selected in the notebook editor** under Session options.

In [ ]:
!nvidia-smi --query-gpu=index,name,memory.total,compute_cap --format=csv
import torch, sys

n = torch.cuda.device_count()
print("torch", torch.__version__, "| cuda", torch.version.cuda, "| devices", n)
for i in range(n):
    cap = torch.cuda.get_device_capability(i)
    print(" ", i, torch.cuda.get_device_name(i), "sm_%d%d" % cap,
          "bf16" if cap >= (8, 0) else "no bf16 -> fp16 + grad scaler")

assert n >= 1, "no GPU at all. Session options -> Accelerator -> GPU T4 x2"
cap0 = torch.cuda.get_device_capability(0)
if cap0 < (7, 0):
    raise SystemExit(
        f"STOP: {torch.cuda.get_device_name(0)} is sm_{cap0[0]}{cap0[1]}, which this "
        f"PyTorch build does not support (needs sm_70+).
"
        f"Set Session options -> Accelerator to 'GPU T4 x2' and re-run.
"
        f"Stopping now rather than after an 8 GB download.")
if n < 2:
    print(f"
NOTE: only {n} GPU visible. Training will run single-process; "
          f"the DDP speed-up needs 'GPU T4 x2'.")


## 2. Dependencies

Pinned to the **same major versions this repo runs locally**, not to floors.
The API moved between transformers 4.x and 5.x in two ways this code depends
on: `from_pretrained(dtype=...)` was `torch_dtype=`, and
`TrainingArguments(eval_strategy=...)` was `evaluation_strategy=`. A `>=4.44`
floor would let Kaggle resolve to 4.x and fail at model load.

In [ ]:
!pip install -q -U "transformers>=5.0,<6" "peft>=0.17" "bitsandbytes>=0.45" \
    "accelerate>=1.0" "datasets>=3.0" 2>&1 | tail -2
import transformers, peft, torch
print("transformers", transformers.__version__, "| peft", peft.__version__,
      "| torch", torch.__version__)
assert int(transformers.__version__.split(".")[0]) >= 5, \
    "this notebook uses the transformers 5.x API (dtype=, eval_strategy=)"


## 3. Data

Mounted read-only from the `agent-loop-tool-calling-traces` dataset.

In [ ]:
from pathlib import Path
DATA = Path("/kaggle/input/agent-loop-tool-calling-traces")
assert DATA.exists(), "attach the 'agent-loop-tool-calling-traces' dataset to this notebook"
for f in sorted(DATA.iterdir()):
    print(f.name, f"{f.stat().st_size/1e6:.2f} MB")


## 4. The training script

Written to disk so `torchrun` can spawn one process per GPU — a notebook cell
cannot be the DDP entry point, since each rank has to import a real module.

Carried as base64 rather than a quoted literal, because the script contains its
own triple-quoted docstrings and a raw triple-quoted wrapper would end at the
first inner docstring.

In [ ]:
import base64, pathlib
SRC_B64 = "CiIiIlFMb1JBIG9uIDJ4IFQ0IHVuZGVyIEREUC4gV3JpdHRlbiBieSBmaW5ldHVuZS9rYWdnbGVfa2VybmVsLnB5LiIiIgppbXBvcnQganNvbiwgb3MsIHN5cywgdGltZQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCmltcG9ydCB0b3JjaApmcm9tIHRvcmNoLnV0aWxzLmRhdGEgaW1wb3J0IERhdGFzZXQKZnJvbSB0cmFuc2Zvcm1lcnMgaW1wb3J0IChBdXRvTW9kZWxGb3JDYXVzYWxMTSwgQXV0b1Rva2VuaXplciwgQml0c0FuZEJ5dGVzQ29uZmlnLAogICAgICAgICAgICAgICAgICAgICAgICAgIFRyYWluZXIsIFRyYWluaW5nQXJndW1lbnRzLCBFYXJseVN0b3BwaW5nQ2FsbGJhY2spCmZyb20gcGVmdCBpbXBvcnQgTG9yYUNvbmZpZywgZ2V0X3BlZnRfbW9kZWwsIHByZXBhcmVfbW9kZWxfZm9yX2tiaXRfdHJhaW5pbmcKCkJBU0UgICAgICA9IG9zLmVudmlyb24uZ2V0KCJCQVNFX01PREVMIiwgIlF3ZW4vUXdlbjMtNEItSW5zdHJ1Y3QtMjUwNyIpCkRBVEFfRElSICA9IFBhdGgob3MuZW52aXJvbi5nZXQoIkRBVEFfRElSIiwgIi9rYWdnbGUvaW5wdXQvYWdlbnQtbG9vcC10b29sLWNhbGxpbmctdHJhY2VzIikpCk9VVCAgICAgICA9IFBhdGgob3MuZW52aXJvbi5nZXQoIk9VVF9ESVIiLCAiL2thZ2dsZS93b3JraW5nL2FkYXB0ZXIiKSkKTUFYX0xFTiAgID0gaW50KG9zLmVudmlyb24uZ2V0KCJNQVhfTEVOIiwgIjIwNDgiKSkKRVBPQ0hTICAgID0gZmxvYXQob3MuZW52aXJvbi5nZXQoIkVQT0NIUyIsICIxIikpCkxPUkFfUiAgICA9IGludChvcy5lbnZpcm9uLmdldCgiTE9SQV9SIiwgIjE2IikpCkxSICAgICAgICA9IGZsb2F0KG9zLmVudmlyb24uZ2V0KCJMUiIsICIyZS00IikpCgpMT0NBTF9SQU5LID0gaW50KG9zLmVudmlyb24uZ2V0KCJMT0NBTF9SQU5LIiwgIjAiKSkKV09STERfU0laRSA9IGludChvcy5lbnZpcm9uLmdldCgiV09STERfU0laRSIsICIxIikpCklTX01BSU4gICAgPSBMT0NBTF9SQU5LID09IDAKCgpkZWYgbG9nKCphKToKICAgIGlmIElTX01BSU46CiAgICAgICAgcHJpbnQoKmEsIGZsdXNoPVRydWUpCgoKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tIGRhdGEKCmRlZiBsb2FkKHBhdGgpOgogICAgcm93cyA9IFtdCiAgICB3aXRoIG9wZW4ocGF0aCwgZW5jb2Rpbmc9InV0Zi04IikgYXMgZmg6CiAgICAgICAgZm9yIGxpbmUgaW4gZmg6CiAgICAgICAgICAgIGxpbmUgPSBsaW5lLnN0cmlwKCkKICAgICAgICAgICAgaWYgbGluZToKICAgICAgICAgICAgICAgIHJvd3MuYXBwZW5kKGpzb24ubG9hZHMobGluZSlbIm1lc3NhZ2VzIl0pCiAgICByZXR1cm4gcm93cwoKCmRlZiBfaWRzKHJlc3VsdCk6CiAgICAiIiJOb3JtYWxpc2UgYXBwbHlfY2hhdF90ZW1wbGF0ZSdzIHJldHVybiB0byBhIGZsYXQgbGlzdCBvZiB0b2tlbiBpZHMuCgogICAgTk9UIG9wdGlvbmFsLiB0cmFuc2Zvcm1lcnMgNC54IHJldHVybnMgbGlzdFtpbnRdOyA1LnggcmV0dXJucyBhCiAgICBCYXRjaEVuY29kaW5nLCB3aGVyZSBsZW4oKSBpcyB0aGUgbnVtYmVyIG9mIEtFWVMgKDIpLCBub3QgdGhlIG51bWJlciBvZgogICAgdG9rZW5zLiBXaXRob3V0IHRoaXMsIHRoZSBzdHJpY3QtcHJlZml4IGd1YXJkIGJlbG93IGNvbXBhcmVzIGRpY3Qgc2xpY2VzLAogICAgZXZlcnkgdHJhY2UgZmFpbHMgaXQsIGVuY29kZSgpIHJldHVybnMgTm9uZSBmb3IgYWxsIG9mIHRoZW0sIGFuZCB0cmFpbmluZwogICAgc3RhcnRzIG9uIGFuIEVNUFRZIGRhdGFzZXQuIFRoZSBmYWlsdXJlIGlzIHNpbGVudCDigJQgdGhlIHJ1biByZXBvcnRzCiAgICAidHJhaW4gMCBldmFsIDAiIGFuZCBwcm9jZWVkcy4KCiAgICBmaW5ldHVuZS9xbG9yYS5weSBjYXJyaWVzIHRoZSBzYW1lIGhlbHBlciBmb3IgdGhlIHNhbWUgcmVhc29uOyB0aGUgdHdvIG11c3QKICAgIG5vdCBkcmlmdCBhcGFydC4KICAgICIiIgogICAgaWRzID0gcmVzdWx0WyJpbnB1dF9pZHMiXSBpZiBoYXNhdHRyKHJlc3VsdCwgImtleXMiKSBlbHNlIHJlc3VsdAogICAgaWYgaWRzIGFuZCBpc2luc3RhbmNlKGlkc1swXSwgbGlzdCk6ICAgICAgICAjIGEgYmF0Y2ggZGltZW5zaW9uIG9uIHNvbWUgdmVyc2lvbnMKICAgICAgICBpZHMgPSBpZHNbMF0KICAgIHJldHVybiBsaXN0KGlkcykKCgpkZWYgZW5jb2RlKHRvaywgbWVzc2FnZXMpOgogICAgIiIiVG9rZW5pc2Ugb25lIHRyYWNlLCBtYXNraW5nIGxvc3MgdG8gYXNzaXN0YW50IHR1cm5zIG9ubHkuCgogICAgVGhpcyBpcyBkZWxpYmVyYXRlbHkgdGhlIFNBTUUgYWxnb3JpdGhtIGFzIGZpbmV0dW5lL3Fsb3JhLnB5OjplbmNvZGUsIG5vdCBhCiAgICByZWltcGxlbWVudGF0aW9uLiBBbiBlYXJsaWVyIHZlcnNpb24gaGVyZSB3YWxrZWQgZXZlcnkgbWVzc2FnZSBhbmQgc2xpY2VkCiAgICBtZXNzYWdlc1s6aV0gZnJvbSBpPTAsIHdoaWNoIHRyYW5zZm9ybWVycyA1LnggcmVqZWN0cyBvdXRyaWdodCAoIkNhbm5vdAogICAgYXBwbHkgY2hhdCB0ZW1wbGF0ZSB0byBhbiBlbXB0eSBjb252ZXJzYXRpb24iKS4gV2Fsa2luZyBvbmx5IGFzc2lzdGFudAogICAgdHVybnMgYXZvaWRzIHRoYXQgYnkgY29uc3RydWN0aW9uLCBiZWNhdXNlIGEgdHJhY2UgYWx3YXlzIG9wZW5zIHdpdGggc3lzdGVtCiAgICBhbmQgdXNlci4KCiAgICBNYXNraW5nIGlzIGV4cGxpY2l0IHJhdGhlciB0aGFuIGRlbGVnYXRlZCB0byBhIHRyYWluZXIgZmxhZzogYSBzaWxlbnQKICAgIG1hc2tpbmcgYnVnIHRyYWlucyB0aGUgbW9kZWwgdG8gcHJlZGljdCB0aGUgc3lzdGVtIHByb21wdCBhbmQgbG9va3MgbGlrZSBhCiAgICBwZXJmZWN0bHkgaGVhbHRoeSBydW4gdW50aWwgdGhlIGFkYXB0ZXIgaXMgdXNlbGVzcy4gQSB0cmFjZSB3aG9zZSB0ZW1wbGF0ZQogICAgZG9lcyBub3QgZXh0ZW5kIGFzIGEgc3RyaWN0IHByZWZpeCBpcyBkcm9wcGVkIHJhdGhlciB0aGFuIHRyYWluZWQgb24gYQogICAgbWlzYWxpZ25tZW50LgogICAgIiIiCiAgICBmdWxsID0gX2lkcyh0b2suYXBwbHlfY2hhdF90ZW1wbGF0ZShtZXNzYWdlcywgdG9rZW5pemU9VHJ1ZSkpCiAgICBpZiBsZW4oZnVsbCkgPiBNQVhfTEVOOgogICAgICAgIHJldHVybiBOb25lCiAgICBsYWJlbHMgPSBbLTEwMF0gKiBsZW4oZnVsbCkKCiAgICBmb3IgaSwgbXNnIGluIGVudW1lcmF0ZShtZXNzYWdlcyk6CiAgICAgICAgaWYgbXNnWyJyb2xlIl0gIT0gImFzc2lzdGFudCI6CiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgcHJvbXB0ID0gX2lkcyh0b2suYXBwbHlfY2hhdF90ZW1wbGF0ZSgKICAgICAgICAgICAgbWVzc2FnZXNbOmldLCB0b2tlbml6ZT1UcnVlLCBhZGRfZ2VuZXJhdGlvbl9wcm9tcHQ9VHJ1ZSkpCiAgICAgICAgdXB0byA9IF9pZHModG9rLmFwcGx5X2NoYXRfdGVtcGxhdGUobWVzc2FnZXNbOmkgKyAxXSwgdG9rZW5pemU9VHJ1ZSkpCiAgICAgICAgaWYgZnVsbFs6bGVuKHByb21wdCldICE9IHByb21wdCBvciBmdWxsWzpsZW4odXB0byldICE9IHVwdG86CiAgICAgICAgICAgIHJldHVybiBOb25lCiAgICAgICAgbGFiZWxzW2xlbihwcm9tcHQpOmxlbih1cHRvKV0gPSBmdWxsW2xlbihwcm9tcHQpOmxlbih1cHRvKV0KCiAgICBpZiBhbGwobCA9PSAtMTAwIGZvciBsIGluIGxhYmVscyk6CiAgICAgICAgcmV0dXJuIE5vbmUKICAgIHJldHVybiB7ImlucHV0X2lkcyI6IGZ1bGwsICJsYWJlbHMiOiBsYWJlbHN9CgoKY2xhc3MgVHJhY2VzKERhdGFzZXQpOgogICAgZGVmIF9faW5pdF9fKHNlbGYsIHJvd3MpOiBzZWxmLnJvd3MgPSByb3dzCiAgICBkZWYgX19sZW5fXyhzZWxmKTogcmV0dXJuIGxlbihzZWxmLnJvd3MpCiAgICBkZWYgX19nZXRpdGVtX18oc2VsZiwgaSk6IHJldHVybiBzZWxmLnJvd3NbaV0KCgpkZWYgY29sbGF0ZShwYWRfaWQpOgogICAgZGVmIGZuKGJhdGNoKToKICAgICAgICBuID0gbWF4KGxlbihiWyJpbnB1dF9pZHMiXSkgZm9yIGIgaW4gYmF0Y2gpCiAgICAgICAgb3V0ID0geyJpbnB1dF9pZHMiOiBbXSwgImxhYmVscyI6IFtdLCAiYXR0ZW50aW9uX21hc2siOiBbXX0KICAgICAgICBmb3IgYiBpbiBiYXRjaDoKICAgICAgICAgICAgcGFkID0gbiAtIGxlbihiWyJpbnB1dF9pZHMiXSkKICAgICAgICAgICAgb3V0WyJpbnB1dF9pZHMiXS5hcHBlbmQoYlsiaW5wdXRfaWRzIl0gKyBbcGFkX2lkXSAqIHBhZCkKICAgICAgICAgICAgb3V0WyJsYWJlbHMiXS5hcHBlbmQoYlsibGFiZWxzIl0gKyBbLTEwMF0gKiBwYWQpCiAgICAgICAgICAgIG91dFsiYXR0ZW50aW9uX21hc2siXS5hcHBlbmQoWzFdICogbGVuKGJbImlucHV0X2lkcyJdKSArIFswXSAqIHBhZCkKICAgICAgICByZXR1cm4ge2s6IHRvcmNoLnRlbnNvcih2LCBkdHlwZT10b3JjaC5sb25nKSBmb3IgaywgdiBpbiBvdXQuaXRlbXMoKX0KICAgIHJldHVybiBmbgoKCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSBtb2RlbAoKZGVmIG1haW4oKToKICAgIGNhcCA9IHRvcmNoLmN1ZGEuZ2V0X2RldmljZV9jYXBhYmlsaXR5KCkKICAgIG5hbWUgPSB0b3JjaC5jdWRhLmdldF9kZXZpY2VfbmFtZSgpCiAgICB2aXNpYmxlID0gdG9yY2guY3VkYS5kZXZpY2VfY291bnQoKQoKICAgICMgUmVwb3J0IHRoZSBBQ1RVQUwgZGV2aWNlIGNvdW50LCBub3QgV09STERfU0laRS4gQW4gZWFybGllciB2ZXJzaW9uIHByaW50ZWQKICAgICMgeHtXT1JMRF9TSVpFfSDigJQgdGhlIHZhbHVlIHRvcmNocnVuIHdhcyB0b2xkIHRvIHVzZSDigJQgc28gYSBzaW5nbGUtR1BVIGJveAogICAgIyBsb2dnZWQgIngyIiBhbmQgdGhlbiBkaWVkIHdpdGggImludmFsaWQgZGV2aWNlIG9yZGluYWwiLiBUaGUgbG9nIGRlc2NyaWJlZAogICAgIyB0aGUgYXNzdW1wdGlvbiwgbm90IHRoZSBtYWNoaW5lLgogICAgIwogICAgIyBiZjE2IG5lZWRzIEFtcGVyZSAoc21fODArKS4gdG9yY2guY3VkYS5pc19iZjE2X3N1cHBvcnRlZCgpIHJldHVybnMgVHJ1ZSBvbgogICAgIyBhIFRlc2xhIFAxMDAgKHNtXzYwKSwgd2hpY2ggaGFzIG5vIGJmMTYgd2hhdHNvZXZlciwgc28gdGhlIGNhcGFiaWxpdHkgaXMKICAgICMgY2hlY2tlZCBkaXJlY3RseSByYXRoZXIgdGhhbiBhc2tlZCBmb3IuCiAgICB1c2VfYmYxNiA9IGNhcCA+PSAoOCwgMCkKICAgIGR0eXBlID0gdG9yY2guYmZsb2F0MTYgaWYgdXNlX2JmMTYgZWxzZSB0b3JjaC5mbG9hdDE2CiAgICBsb2coZiJbZ3B1XSB7bmFtZX0gc21fe2NhcFswXX17Y2FwWzFdfSB4e3Zpc2libGV9IHZpc2libGUgIgogICAgICAgIGYiKHdvcmxkX3NpemUge1dPUkxEX1NJWkV9KSAgcHJlY2lzaW9uPSIKICAgICAgICBmInsnYmYxNicgaWYgdXNlX2JmMTYgZWxzZSAnZnAxNiArIGdyYWQgc2NhbGVyJ30iKQoKICAgIGlmIGNhcCA8ICg3LCAwKToKICAgICAgICByYWlzZSBTeXN0ZW1FeGl0KAogICAgICAgICAgICBmIkFCT1JUOiB7bmFtZX0gaXMgc21fe2NhcFswXX17Y2FwWzFdfS4gVGhlIEthZ2dsZSBQeVRvcmNoIGJ1aWxkICIKICAgICAgICAgICAgZiJzdXBwb3J0cyBzbV83MCBhbmQgYWJvdmUsIHNvIHRoaXMgR1BVIGNhbm5vdCBydW4gdGhlIHRyYWluaW5nICIKICAgICAgICAgICAgZiJhdCBhbGwuXG5cbiIKICAgICAgICAgICAgZiJLYWdnbGUgYXNzaWduZWQgYSBkZWZhdWx0IGFjY2VsZXJhdG9yLiBJbiB0aGUgbm90ZWJvb2sgZWRpdG9yIHNldCAiCiAgICAgICAgICAgIGYiU2Vzc2lvbiBvcHRpb25zIC0+IEFjY2VsZXJhdG9yIHRvICdHUFUgVDQgeDInIGFuZCByZS1ydW4uIFRoZSBBUEkgIgogICAgICAgICAgICBmImNhbm5vdCByZXF1ZXN0IGl0OiBtYWNoaW5lX3NoYXBlIHNpbGVudGx5IG5vcm1hbGlzZXMgYW55IHVua25vd24gIgogICAgICAgICAgICBmInZhbHVlIGJhY2sgdG8gYSBnZW5lcmljICdHcHUnLiIpCgogICAgaWYgV09STERfU0laRSA+IHZpc2libGU6CiAgICAgICAgcmFpc2UgU3lzdGVtRXhpdCgKICAgICAgICAgICAgZiJBQk9SVDogbGF1bmNoZWQgd2l0aCB3b3JsZF9zaXplIHtXT1JMRF9TSVpFfSBidXQgb25seSB7dmlzaWJsZX0gR1BVKHMpICIKICAgICAgICAgICAgZiJhcmUgdmlzaWJsZS4gRWFjaCBERFAgcmFuayBwaW5zIGEgd2hvbGUgbW9kZWwgdG8gaXRzIG93biBjYXJkLCBzbyAiCiAgICAgICAgICAgIGYicmFuayB7dmlzaWJsZX0gd291bGQgaW5kZXggYSBkZXZpY2UgdGhhdCBkb2VzIG5vdCBleGlzdCAiCiAgICAgICAgICAgIGYiKCdpbnZhbGlkIGRldmljZSBvcmRpbmFsJykuIikKCiAgICB0b2sgPSBBdXRvVG9rZW5pemVyLmZyb21fcHJldHJhaW5lZChCQVNFKQogICAgaWYgdG9rLnBhZF90b2tlbl9pZCBpcyBOb25lOgogICAgICAgIHRvay5wYWRfdG9rZW4gPSB0b2suZW9zX3Rva2VuCgogICAgdHJhaW5fcm93cyA9IFtyIGZvciByIGluIChlbmNvZGUodG9rLCBtKSBmb3IgbSBpbiBsb2FkKERBVEFfRElSIC8gInRyYWluLmpzb25sIikpIGlmIHJdCiAgICBldmFsX3Jvd3MgID0gW3IgZm9yIHIgaW4gKGVuY29kZSh0b2ssIG0pIGZvciBtIGluIGxvYWQoREFUQV9ESVIgLyAiZXZhbC5qc29ubCIpKSAgaWYgcl0KICAgIGxvZyhmIltkYXRhXSB0cmFpbiB7bGVuKHRyYWluX3Jvd3MpfSAgZXZhbCB7bGVuKGV2YWxfcm93cyl9ICBtYXhfbGVuIHtNQVhfTEVOfSIpCiAgICByYXdfdHJhaW4gPSBsZW4obG9hZChEQVRBX0RJUiAvICJ0cmFpbi5qc29ubCIpKQogICAgaWYgbm90IHRyYWluX3Jvd3Mgb3Igbm90IGV2YWxfcm93czoKICAgICAgICByYWlzZSBTeXN0ZW1FeGl0KAogICAgICAgICAgICBmIkFCT1JUOiBlbmNvZGluZyBwcm9kdWNlZCB7bGVuKHRyYWluX3Jvd3MpfSB0cmFpbiAvIHtsZW4oZXZhbF9yb3dzKX0gZXZhbCByb3dzICIKICAgICAgICAgICAgZiJmcm9tIHtyYXdfdHJhaW59IHRyYWNlcy4gRXZlcnkgdHJhY2UgZmFpbGVkIHRoZSBzdHJpY3QtcHJlZml4IGNoZWNrLCB3aGljaCAiCiAgICAgICAgICAgIGYidXN1YWxseSBtZWFucyBhcHBseV9jaGF0X3RlbXBsYXRlIHJldHVybmVkIGEgQmF0Y2hFbmNvZGluZyBhbmQgX2lkcygpIGlzIG1pc3NpbmcuIikKICAgIGlmIGxlbih0cmFpbl9yb3dzKSA8IHJhd190cmFpbiAqIDAuOToKICAgICAgICBsb2coZiIhISBXQVJOSU5HIGtlcHQgb25seSB7bGVuKHRyYWluX3Jvd3MpfS97cmF3X3RyYWlufSB0cmFjZXMgIgogICAgICAgICAgICBmIih7bGVuKHRyYWluX3Jvd3MpL3Jhd190cmFpbjouMCV9KSDigJQgcmFpc2UgTUFYX0xFTiBvciBjaGVjayB0aGUgY2hhdCB0ZW1wbGF0ZS4iKQoKICAgIHF1YW50ID0gQml0c0FuZEJ5dGVzQ29uZmlnKAogICAgICAgIGxvYWRfaW5fNGJpdD1UcnVlLAogICAgICAgIGJuYl80Yml0X3F1YW50X3R5cGU9Im5mNCIsCiAgICAgICAgYm5iXzRiaXRfdXNlX2RvdWJsZV9xdWFudD1UcnVlLAogICAgICAgIGJuYl80Yml0X2NvbXB1dGVfZHR5cGU9ZHR5cGUsCiAgICApCgogICAgIyBUSEUgRERQIFRSQVA6IHBpbiBhIHdob2xlIG1vZGVsIHRvIFRISVMgcmFuaydzIGNhcmQuIGRldmljZV9tYXA9ImF1dG8iCiAgICAjIHdvdWxkIHNoYXJkIG9uZSBtb2RlbCBhY3Jvc3MgYm90aCBHUFVzIGFuZCB0aGVuIEREUCB3b3VsZCB3cmFwIHRoZSBzaGFyZC4KICAgIG1vZGVsID0gQXV0b01vZGVsRm9yQ2F1c2FsTE0uZnJvbV9wcmV0cmFpbmVkKAogICAgICAgIEJBU0UsCiAgICAgICAgcXVhbnRpemF0aW9uX2NvbmZpZz1xdWFudCwKICAgICAgICBkZXZpY2VfbWFwPXsiIjogTE9DQUxfUkFOS30sCiAgICAgICAgYXR0bl9pbXBsZW1lbnRhdGlvbj0ic2RwYSIsICAgICAgIyBGbGFzaEF0dGVudGlvbi0yIG5lZWRzIHNtXzgwKwogICAgICAgIGR0eXBlPWR0eXBlLAogICAgKQogICAgbW9kZWwuY29uZmlnLnVzZV9jYWNoZSA9IEZhbHNlCiAgICBtb2RlbCA9IHByZXBhcmVfbW9kZWxfZm9yX2tiaXRfdHJhaW5pbmcobW9kZWwsIHVzZV9ncmFkaWVudF9jaGVja3BvaW50aW5nPVRydWUpCgogICAgIyBwcmVwYXJlX21vZGVsX2Zvcl9rYml0X3RyYWluaW5nIHVwY2FzdHMgZXZlcnkgZnJvemVuIG5vcm0gYW5kIGVtYmVkZGluZyB0bwogICAgIyBmcDMyLiBPbiBRd2VuMyB0aGUgdm9jYWJ1bGFyeSBpcyAxNTEsOTM2LCBzbyBsbV9oZWFkIGFsb25lIGlzIDM4OU0gcGFyYW1zCiAgICAjIOKAlCAxLjU1IEdCIGluIGZwMzIsIGZvciB3ZWlnaHRzIExvUkEgbmV2ZXIgdG91Y2hlcy4gUmVjYXN0IHRoZW0uCiAgICBmb3Igbl8sIHAgaW4gbW9kZWwubmFtZWRfcGFyYW1ldGVycygpOgogICAgICAgIGlmIG5vdCBwLnJlcXVpcmVzX2dyYWQgYW5kIHAuZHR5cGUgPT0gdG9yY2guZmxvYXQzMjoKICAgICAgICAgICAgcC5kYXRhID0gcC5kYXRhLnRvKGR0eXBlKQoKICAgIG1vZGVsID0gZ2V0X3BlZnRfbW9kZWwobW9kZWwsIExvcmFDb25maWcoCiAgICAgICAgcj1MT1JBX1IsIGxvcmFfYWxwaGE9TE9SQV9SICogMiwgbG9yYV9kcm9wb3V0PTAuMDUsIGJpYXM9Im5vbmUiLAogICAgICAgIHRhc2tfdHlwZT0iQ0FVU0FMX0xNIiwKICAgICAgICB0YXJnZXRfbW9kdWxlcz1bInFfcHJvaiIsICJrX3Byb2oiLCAidl9wcm9qIiwgIm9fcHJvaiIsCiAgICAgICAgICAgICAgICAgICAgICAgICJnYXRlX3Byb2oiLCAidXBfcHJvaiIsICJkb3duX3Byb2oiXSwKICAgICkpCiAgICBpZiBJU19NQUlOOgogICAgICAgIG1vZGVsLnByaW50X3RyYWluYWJsZV9wYXJhbWV0ZXJzKCkKCiAgICBhcmdzID0gVHJhaW5pbmdBcmd1bWVudHMoCiAgICAgICAgb3V0cHV0X2Rpcj0iL2thZ2dsZS93b3JraW5nL2NrcHQiLAogICAgICAgIG51bV90cmFpbl9lcG9jaHM9RVBPQ0hTLAogICAgICAgICMgQkFUQ0ggU0laRSBJUyBOT1QgMSBIRVJFLCBBTkQgVEhBVCBJUyBUSEUgUE9JTlQuCiAgICAgICAgIyBmaW5ldHVuZS9xbG9yYS5weSB1c2VzIDEgYmVjYXVzZSBvbiB0aGUgNiBHQiBSVFggNDA1MCBWUkFNIGlzIHRoZQogICAgICAgICMgYmluZGluZyBjb25zdHJhaW50IChBR0VOVFMubWQgwqc0LjEpIOKAlCBpdHMgUkVBRE1FIHNheXMgc28gZXhwbGljaXRseS4KICAgICAgICAjIEEgVDQgaGFzIDE2IEdCIGFuZCB0aGF0IGNvbnN0cmFpbnQgc2ltcGx5IGRvZXMgbm90IGFwcGx5LiBDYXJyeWluZyB0aGUKICAgICAgICAjIDEgYWNyb3NzIHdhcyBpbmhlcml0aW5nIGEgbGltaXQgZnJvbSB0aGUgd3JvbmcgbWFjaGluZS4KICAgICAgICAjCiAgICAgICAgIyBCdWRnZXQgYXQgb3VyIG1lYXN1cmVkIG1heCBzZXF1ZW5jZSBvZiAxNTgzIHRva2Vucywgdm9jYWIgMTUxLDkzNiwKICAgICAgICAjIGZwMTYgbG9naXRzOiB3ZWlnaHRzIDIuOCBHQiArIGxvZ2l0cyAoYiB4IDE1ODMgeCAxNTE5MzYgeCAyIEIsIHBsdXMgYQogICAgICAgICMgYmFja3dhcmQgY29weSkgKyBjaGVja3BvaW50ZWQgYWN0aXZhdGlvbnMuCiAgICAgICAgIyAgICAgYj0xICB+NC41IEdCICAgICAgYj00ICB+Ny43IEdCICAgICAgYj04ICB+MTEuNSBHQgogICAgICAgICMgNCBsZWF2ZXMgY29tZm9ydGFibGUgaGVhZHJvb20gb24gMTYgR0IgYW5kIHN0b3BzIHN0YXJ2aW5nIHRoZSB0ZW5zb3IKICAgICAgICAjIGNvcmVzLCB3aGljaCBpcyB3aGF0IGJhdGNoIDEgd2FzIGRvaW5nLgogICAgICAgIHBlcl9kZXZpY2VfdHJhaW5fYmF0Y2hfc2l6ZT1pbnQob3MuZW52aXJvbi5nZXQoIkJBVENIIiwgIjQiKSksCiAgICAgICAgcGVyX2RldmljZV9ldmFsX2JhdGNoX3NpemU9NCwKICAgICAgICAjIEdsb2JhbCBiYXRjaCBzdGF5cyAxNiAoNCB4IDIgYWNjdW0geCAyIGNhcmRzKSwgaWRlbnRpY2FsIHRvIHRoZSBvbGQKICAgICAgICAjIDEgeCA4IHggMiDigJQgc28gdGhpcyBpcyBhIHRocm91Z2hwdXQgY2hhbmdlLCBub3QgYSBoeXBlcnBhcmFtZXRlcgogICAgICAgICMgY2hhbmdlLCBhbmQgdGhlIGxvc3MgY3VydmUgc3RheXMgY29tcGFyYWJsZSB0byBwcmV2aW91cyBydW5zLgogICAgICAgIGdyYWRpZW50X2FjY3VtdWxhdGlvbl9zdGVwcz0yLAogICAgICAgICMgU2VxdWVuY2VzIHJ1biA5OTMgdG9rZW5zIG1lZGlhbiBhZ2FpbnN0IGEgMTU4MyBtYXguIFJhbmRvbSBiYXRjaGluZwogICAgICAgICMgcGFkcyBldmVyeSBzZXF1ZW5jZSB0byBpdHMgYmF0Y2gncyBsb25nZXN0LCBzbyBhIHNob3J0IHRyYWNlIGJhdGNoZWQKICAgICAgICAjIHdpdGggYSBsb25nIG9uZSB3YXN0ZXMgbW9zdCBvZiBpdHMgY29tcHV0ZSBvbiBwYWRkaW5nLiBHcm91cGluZyBieQogICAgICAgICMgbGVuZ3RoIG1ha2VzIGJhdGNoZXMgaG9tb2dlbmVvdXMgYW5kIGlzIGZyZWUuCiAgICAgICAgZ3JvdXBfYnlfbGVuZ3RoPVRydWUsCiAgICAgICAgZ3JhZGllbnRfY2hlY2twb2ludGluZz1UcnVlLAogICAgICAgIGdyYWRpZW50X2NoZWNrcG9pbnRpbmdfa3dhcmdzPXsidXNlX3JlZW50cmFudCI6IEZhbHNlfSwKICAgICAgICBsZWFybmluZ19yYXRlPUxSLAogICAgICAgIGxyX3NjaGVkdWxlcl90eXBlPSJjb3NpbmUiLAogICAgICAgIHdhcm11cF9yYXRpbz0wLjAzLAogICAgICAgIGxvZ2dpbmdfc3RlcHM9MTAsCiAgICAgICAgZXZhbF9zdHJhdGVneT0ic3RlcHMiLAogICAgICAgIGV2YWxfc3RlcHM9MTAsCiAgICAgICAgc2F2ZV9zdHJhdGVneT0ic3RlcHMiLAogICAgICAgIHNhdmVfc3RlcHM9MTAsCiAgICAgICAgc2F2ZV90b3RhbF9saW1pdD0yLAogICAgICAgIGxvYWRfYmVzdF9tb2RlbF9hdF9lbmQ9VHJ1ZSwgICAgICAgIyBiZXN0IGJ5IEVWQUwgbG9zcywgbm90IHRoZSBsYXN0IHN0ZXAKICAgICAgICBtZXRyaWNfZm9yX2Jlc3RfbW9kZWw9ImV2YWxfbG9zcyIsCiAgICAgICAgZ3JlYXRlcl9pc19iZXR0ZXI9RmFsc2UsCiAgICAgICAgYmYxNj11c2VfYmYxNiwKICAgICAgICBmcDE2PW5vdCB1c2VfYmYxNiwKICAgICAgICBvcHRpbT0icGFnZWRfYWRhbXdfOGJpdCIsCiAgICAgICAgcmVwb3J0X3RvPVtdLCAgICAgICAgICAgICAgICAgICAgICAjIG5vIHdhbmRiLCBubyB0ZWxlbWV0cnkgKMKnMi4xKQogICAgICAgIGRkcF9maW5kX3VudXNlZF9wYXJhbWV0ZXJzPUZhbHNlLCAgIyBMb1JBICsgY2hlY2twb2ludGluZzogcmVxdWlyZWQKICAgICAgICBkYXRhbG9hZGVyX251bV93b3JrZXJzPTIsCiAgICAgICAgc2VlZD0yNjExNywKICAgICkKCiAgICB0cmFpbmVyID0gVHJhaW5lcigKICAgICAgICBtb2RlbD1tb2RlbCwgYXJncz1hcmdzLAogICAgICAgIHRyYWluX2RhdGFzZXQ9VHJhY2VzKHRyYWluX3Jvd3MpLAogICAgICAgIGV2YWxfZGF0YXNldD1UcmFjZXMoZXZhbF9yb3dzKSwKICAgICAgICBkYXRhX2NvbGxhdG9yPWNvbGxhdGUodG9rLnBhZF90b2tlbl9pZCksCiAgICAgICAgY2FsbGJhY2tzPVtFYXJseVN0b3BwaW5nQ2FsbGJhY2soZWFybHlfc3RvcHBpbmdfcGF0aWVuY2U9MyldLAogICAgKQoKICAgIHQwID0gdGltZS50aW1lKCkKICAgIHRyYWluZXIudHJhaW4oKQogICAgZWxhcHNlZCA9IHRpbWUudGltZSgpIC0gdDAKCiAgICBpZiBJU19NQUlOOgogICAgICAgIE9VVC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICAgICAgbW9kZWwuc2F2ZV9wcmV0cmFpbmVkKHN0cihPVVQpKQogICAgICAgIHRvay5zYXZlX3ByZXRyYWluZWQoc3RyKE9VVCkpCiAgICAgICAgaGlzdCA9IFtoIGZvciBoIGluIHRyYWluZXIuc3RhdGUubG9nX2hpc3RvcnkgaWYgImV2YWxfbG9zcyIgaW4gaF0KICAgICAgICBiZXN0ID0gbWluKChoWyJldmFsX2xvc3MiXSBmb3IgaCBpbiBoaXN0KSwgZGVmYXVsdD1mbG9hdCgibmFuIikpCiAgICAgICAgbGFzdF90cmFpbiA9IG5leHQoKGhbImxvc3MiXSBmb3IgaCBpbiByZXZlcnNlZCh0cmFpbmVyLnN0YXRlLmxvZ19oaXN0b3J5KQogICAgICAgICAgICAgICAgICAgICAgICAgICBpZiAibG9zcyIgaW4gaCksIGZsb2F0KCJuYW4iKSkKICAgICAgICBwZWFrID0gbWF4KHRvcmNoLmN1ZGEubWF4X21lbW9yeV9hbGxvY2F0ZWQoaSkgLyAxZTkKICAgICAgICAgICAgICAgICAgIGZvciBpIGluIHJhbmdlKHRvcmNoLmN1ZGEuZGV2aWNlX2NvdW50KCkpKQogICAgICAgIHN1bW1hcnkgPSB7CiAgICAgICAgICAgICJlbGFwc2VkX3MiOiByb3VuZChlbGFwc2VkLCAxKSwKICAgICAgICAgICAgInBlYWtfZ2JfcGVyX2NhcmQiOiByb3VuZChwZWFrLCAyKSwKICAgICAgICAgICAgIndvcmxkX3NpemUiOiBXT1JMRF9TSVpFLAogICAgICAgICAgICAiYmVzdF9ldmFsX2xvc3MiOiBiZXN0LAogICAgICAgICAgICAiZmluYWxfdHJhaW5fbG9zcyI6IGxhc3RfdHJhaW4sCiAgICAgICAgICAgICJ0cmFpbl90cmFjZXMiOiBsZW4odHJhaW5fcm93cyksCiAgICAgICAgICAgICJldmFsX3RyYWNlcyI6IGxlbihldmFsX3Jvd3MpLAogICAgICAgICAgICAibWF4X2xlbiI6IE1BWF9MRU4sICJlcG9jaHMiOiBFUE9DSFMsICJsb3JhX3IiOiBMT1JBX1IsCiAgICAgICAgfQogICAgICAgIFBhdGgoIi9rYWdnbGUvd29ya2luZy9zdW1tYXJ5Lmpzb24iKS53cml0ZV90ZXh0KGpzb24uZHVtcHMoc3VtbWFyeSwgaW5kZW50PTIpKQogICAgICAgIGxvZygiXG4iICsganNvbi5kdW1wcyhzdW1tYXJ5LCBpbmRlbnQ9MikpCgogICAgICAgICMgVGhlIGRpYWdub3NpcyB0aGF0IHRoZSBwcmV2aW91cyB0aHJlZSBhZGFwdGVycyBmYWlsZWQgb24uIEEgdHJhaW5pbmcKICAgICAgICAjIGxvc3MgdGhpcyBsb3cgbWVhbnMgdGhlIG1vZGVsIHJlcHJvZHVjZXMgaXRzIHRyYWNlcyBuZWFyLWV4YWN0bHksIGFuZAogICAgICAgICMgZXZlcnkgcHJveHkgbWV0cmljIHdpbGwgbG9vayBleGNlbGxlbnQgd2hpbGUgZW5kLXRvLWVuZCBhY2N1cmFjeQogICAgICAgICMgY29sbGFwc2VzLiBTYXkgc28gaGVyZSByYXRoZXIgdGhhbiBkaXNjb3ZlcmluZyBpdCBpbiB0aGUgZ2F0ZS4KICAgICAgICBpZiBsYXN0X3RyYWluID09IGxhc3RfdHJhaW4gYW5kIGxhc3RfdHJhaW4gPCAwLjA1OgogICAgICAgICAgICBsb2coZiJcbiEhIFdBUk5JTkcgdHJhaW4gbG9zcyB7bGFzdF90cmFpbjouNGZ9IGluZGljYXRlcyBNRU1PUklTQVRJT04uIgogICAgICAgICAgICAgICAgZiJcbiEhIHYyIGFuZCB2MyBib3RoIGxvb2tlZCBsaWtlIHRoaXMgYW5kIGJvdGggc2NvcmVkIHdvcnNlIHRoYW4gYmFzZS4iCiAgICAgICAgICAgICAgICBmIlxuISEgUHJlZmVyIHRoZSBiZXN0LWV2YWwgY2hlY2twb2ludCBhbmQgY29uc2lkZXIgbW9yZSBkYXRhLCBub3QgbW9yZSBlcG9jaHMuIikKICAgICAgICBpZiBiZXN0ID09IGJlc3QgYW5kIGxhc3RfdHJhaW4gPT0gbGFzdF90cmFpbiBhbmQgYmVzdCA+IGxhc3RfdHJhaW4gKiA1OgogICAgICAgICAgICBsb2coZiJcbiEhIFdBUk5JTkcgZXZhbCBsb3NzIHtiZXN0Oi40Zn0gPj4gdHJhaW4gbG9zcyB7bGFzdF90cmFpbjouNGZ9OiBvdmVyZml0dGluZy4iKQoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBtYWluKCkK"
src = base64.b64decode(SRC_B64).decode("utf-8")
pathlib.Path("/kaggle/working/train_ddp.py").write_text(src, encoding="utf-8")
compile(src, "train_ddp.py", "exec")   # fail here, not inside torchrun
print("wrote train_ddp.py", len(src), "bytes")


## 5. Train on both GPUs

`--nproc_per_node=2` starts one process per card. Each holds a full 4-bit copy
(~2.8 GB of 16 GB) and DDP all-reduces the LoRA gradients.

In [ ]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["BASE_MODEL"] = "Qwen/Qwen3-4B-Instruct-2507"
os.environ["DATA_DIR"]   = "/kaggle/input/agent-loop-tool-calling-traces"
os.environ["MAX_LEN"]    = "2048"
os.environ["EPOCHS"]     = "1"
os.environ["BATCH"]      = "4"

# nproc is DETECTED, not assumed. The previous version hardcoded 2 and Kaggle
# handed the run a single P100, so rank 1 indexed a GPU that did not exist and
# torchrun died after a 40 GB model download.
import torch
NPROC = max(1, torch.cuda.device_count())
print(f"launching torchrun with --nproc_per_node={NPROC} "
      f"({NPROC} GPU(s) detected: "
      f"{[torch.cuda.get_device_name(i) for i in range(NPROC)]})")
if NPROC == 1:
    print("NOTE: single GPU. This still trains, just without the DDP speed-up.")

!cd /kaggle/working && torchrun --nproc_per_node=$NPROC --master_port=29500 train_ddp.py


## 6. Training result

In [ ]:
import json, pathlib
p = pathlib.Path("/kaggle/working/summary.json")
print(json.dumps(json.loads(p.read_text()), indent=2) if p.exists()
      else "no summary - training did not finish")
!ls -la /kaggle/working/adapter 2>/dev/null || echo "no adapter written"


## 7. Benchmark

Scores the trained adapter against the untouched base on **240 held-out tasks**
over 40 documents generated with a different seed from the training corpus.
Zero overlap, asserted in the repo's own tests.

This replaces a gate that scored **one task over eight trials**. That gate
measured the same base model at 7/8 in one session and 5/8 in another, so it
could not separate a real regression from sampling noise — and three adapters
were judged on it.

`finetune/benchmark.py` is shipped in verbatim rather than reimplemented, so
these numbers are directly comparable to a local run.

In [ ]:
import base64, pathlib, sys
BENCH_B64 = "..."
pathlib.Path("/kaggle/working/shipped.py").write_text(
    base64.b64decode(BENCH_B64).decode("utf-8"), encoding="utf-8")
sys.path.insert(0, "/kaggle/working")
print("benchmark harness ready")


In [ ]:
import sys, json, pathlib, torch
sys.path.insert(0, "/kaggle/working")
from shipped import build_tasks, run_benchmark, report, wilson, separated
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

BASE = "Qwen/Qwen3-4B-Instruct-2507"
tok = AutoTokenizer.from_pretrained(BASE)
if tok.pad_token_id is None:
    tok.pad_token = tok.eos_token

def loader(adapter=None):
    m = AutoModelForCausalLM.from_pretrained(
        BASE, device_map={"": 0}, dtype=torch.float16, attn_implementation="sdpa")
    if adapter:
        m = PeftModel.from_pretrained(m, adapter).merge_and_unload()
    m.eval()
    return m

def make_generate(model):
    def gen(messages):
        ids = tok.apply_chat_template(messages, tokenize=True,
                                      add_generation_prompt=True, return_tensors="pt")
        ids = ids.to(model.device)
        with torch.no_grad():
            out = model.generate(ids, max_new_tokens=320, temperature=0.2,
                                 do_sample=True, pad_token_id=tok.pad_token_id)
        return tok.decode(out[0][ids.shape[-1]:], skip_special_tokens=True)
    return gen

TASKS = build_tasks()
RUNS = 3
print(f"{len(TASKS)} held-out tasks x {RUNS} runs = {len(TASKS)*RUNS} samples per model")


In [ ]:
base_model = loader(None)
base_res = run_benchmark("base " + BASE, make_generate(base_model), TASKS, RUNS)
del base_model; torch.cuda.empty_cache()
print(f"base accuracy {base_res.accuracy:.1%}")


In [ ]:
tuned = loader("/kaggle/working/adapter")
cand_res = run_benchmark("adapter (this run)", make_generate(tuned), TASKS, RUNS)
del tuned; torch.cuda.empty_cache()
print(f"adapter accuracy {cand_res.accuracy:.1%}")


## 8. Verdict, and the file to copy back into the repo

`separated` is the honest bar: if the two 95% intervals overlap, this benchmark
**cannot tell the models apart**, and that is the result — not a rounding
decision in favour of whichever number is larger.

In [ ]:
md = report(cand_res, base_res)
print(md)

payload = {
    "source": "kaggle t4x2",
    "base": {"ref": base_res.ref, "accuracy": base_res.accuracy,
             "ci95": wilson(base_res.correct, base_res.n),
             "tally": base_res.tally(), "samples": base_res.n},
    "candidate": {"ref": cand_res.ref, "accuracy": cand_res.accuracy,
                  "ci95": wilson(cand_res.correct, cand_res.n),
                  "tally": cand_res.tally(), "samples": cand_res.n},
    "by_category": {k: {"correct": c, "n": n}
                    for k, (c, n) in cand_res.by_category().items()},
    "separated": separated(cand_res, base_res),
}
pathlib.Path("/kaggle/working/benchmark.json").write_text(json.dumps(payload, indent=2))
pathlib.Path("/kaggle/working/benchmark.md").write_text(md, encoding="utf-8")

print("\n" + ("SEPARATED - the difference exceeds sampling noise"
      if payload["separated"] else
      "NOT SEPARATED - intervals overlap; this benchmark cannot tell them apart"))
print("\nDownload benchmark.json + benchmark.md and run, in the repo:")
print("  python -m finetune.import_benchmark <path-to-benchmark.json>")


## 9. Next

Download `/kaggle/working/adapter/` and run the local publish chain:

```powershell
python -m finetune.publish --model driver     # merge -> GGUF -> q4_K_M -> ollama
```

**Do not register a tag in `config/models.yaml` until the benchmark separates
from base.** Every proxy metric said v2 was excellent; end-to-end said 0/8.